# 対象の動詞は上下の移動を表すか

上から順にセルを実行し、規則を変え、分類器を学習させて、不一致を一つ読みます。暫定ラベルはQwen3.5-9Bによるものです。上下の移動が起きる・起きた場合はCT+、否定される場合はCT-、起きるか示されていない場合はUu、上下の移動を表さない場合はNMです。

### 教材データを読み込む
この準備セルを一度実行してください。授業サイトから本文、暫定ラベル、保存済みの埋め込みを読み込み、箇所のIDで対応付けます。

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = pd.read_json(bundle.open("class7/passages.jsonl"), lines=True).set_index("id")
    labels = pd.read_json(bundle.open("class7/llm-labels.jsonl"), lines=True).set_index("id")
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vectors = pd.DataFrame(saved["vectors"].astype("float32"), index=saved["ids"])

passages["reference"] = labels["label"]

### 1. 予測の前に読む
【対象語】と前後の文を読んでください。何が落ちるのでしょうか。次に進む前にラベルを一つ選んでください。

In [ ]:
passage = passages.loc["dev-0054"]
print(passage["previous"])
print(passage["left"] + "【" + passage["target"] + "】" + passage["right"])
print(passage["next"])

動詞の直前が手掛かりの表現で終わる場合はNM、それ以外はCT+と予測する規則です。結果を予想してから実行し、手掛かりを「声が」に変えて再実行してください。新しい手掛かりでも拾えない用例はどんなものでしょうか。

In [ ]:
cue = "眠りに"
if passage["left"].endswith(cue):
    prediction = "NM"
else:
    prediction = "CT+"
print(prediction, passage["reference"])

### 2. 学習用の作品で分類器を作る
`vectors`の各行が一つの箇所の埋め込みです。420箇所を使って分類器を学習させます。調整用と評価用には別の作品を使います。

In [ ]:
train = passages[passages["split"] == "train"]
dev = passages[passages["split"] == "dev"].dropna(subset=["reference"])
test = passages[passages["split"] == "test"]

train_vectors = vectors.loc[train.index]
train_labels = train["reference"]
model = LogisticRegression(max_iter=2000)
model.fit(train_vectors, train_labels)

暫定ラベルを読める調整用59箇所で一致を確かめます。行が暫定ラベル、列が予測です。Uuのうち何件を拾えたか、評価用の結果を見る前に書いてください。

In [ ]:
dev_predictions = model.predict(vectors.loc[dev.index])
pd.crosstab(dev["reference"], dev_predictions, colnames=["prediction"])

### 3. 評価用の作品で不一致を読む
一致数を確かめてから、最初の不一致を読みます。自分の判断の根拠になる語句を挙げてください。`0`を`1`に変えると次の不一致を読めます。

In [ ]:
test_predictions = model.predict(vectors.loc[test.index])
results = test.assign(prediction=test_predictions)
agrees = results["prediction"] == results["reference"]
print(agrees.sum(), "/", len(results))
disagreements = results[~agrees]

In [ ]:
case = disagreements.iloc[0]
print(case["title"], case["reference"], case["prediction"])
print(case["previous"])
print(case["left"] + "【" + case["target"] + "】" + case["right"])
print(case["next"])